# LLM-assisted classification of the abstracts (prompt version 2.1)

Each abstract retained by the screening (`data/articles/screened_articles.csv`, n = 1,600) is coded on the two dimensions of the strategic AI framework of Huang and Rust (2021): the **marketing stage** that the AI supports (research, strategy or action, with the 4Ps/4Cs area for actions) and the **AI intelligence** by its purpose (mechanical, thinking or feeling). Together they give the nine strategic elements of Table 1 of the article. The prompt (`mdm/prompt.py`) holds the definitions, the decision rules R1-R7, the reference applications of Tables 2 and 3 of the article and 13 worked examples.

**How the calls are made** (`mdm/classify.py`, `mdm/llm_client.py`)
- The API and the model are set only in the configuration cell below; every row of the output keeps the provider, the model and the exact model version returned by the API. Batches saved by a different provider or model are not mixed in: the run stops and asks for them to be moved.
- Several abstracts per request. The model returns `{"results": [...]}`, one entry per study id, constrained by a JSON schema with the allowed labels. Studies left out of an answer are sent again in smaller batches.
- Gemini: temperature 0 and a fixed pause between requests; on a rate limit the client waits the time the API asks for; if the free-tier daily quota ends, the run stops and continues from the next batch when the cell is run again. OpenAI (reasoning models): only the default temperature is accepted, so it is omitted, and the reasoning effort is set in the configuration cell.
- Every batch is saved as soon as it is done.

**API key**: put `LLM_API_KEY=...` in a file named `.env` in the repository folder (it is in `.gitignore`). Never write the key in the notebook.

In [30]:
import pandas as pd
from mdm import classify, prompt
from mdm.llm_client import load_api_key

# ---- LLM used for the classification -------------------------------------------------------------
# OpenAI API (Chat Completions). To use another model, change MODEL; to use another API, change PROVIDER
# (a new API also needs its request method in mdm/llm_client.py). Nothing else in the project names the model.
PROVIDER = 'openai'
MODEL = 'gpt-5.4-nano'
TEMPERATURE = None        # reasoning models accept only the default temperature (use 0.0 with Gemini)
REASONING_EFFORT = 'low'  # OpenAI reasoning models: 'minimal', 'low', 'medium' or 'high'
MIN_INTERVAL = 2          # seconds between requests
FULL_BATCH = 25           # abstracts per request

load_api_key()            # stops here if LLM_API_KEY is missing from .env
print(f'{PROVIDER} / {MODEL}: API key found')

openai / gpt-5.4-nano: API key found


### The prompt

In [31]:
system, phash = classify.system_prompt()
print(f'Prompt version {prompt.PROMPT_VERSION}, hash {phash}, {len(system):,} characters (about {len(system) // 4:,} tokens)')
print(f'Saved in data/framework/classification_prompt_v{prompt.PROMPT_VERSION}.txt')

Prompt version 2.1, hash 0e2ea495ba43, 42,949 characters (about 10,737 tokens)
Saved in data/framework/classification_prompt_v2.1.txt


### Classification of all screened records (1,600)

In [34]:
full = classify.run('full', PROVIDER, MODEL, FULL_BATCH, temperature=TEMPERATURE, min_interval=MIN_INTERVAL,
                 reasoning_effort=REASONING_EFFORT)

full: 1600 records, 64 batches of up to 25; openai / gpt-5.4-nano; prompt v2.1 (0e2ea495ba43)
  batch 1/64: 25 records, 1 call(s), 23,109 tokens, 36s elapsed
  batch 2/64: 25 records, 1 call(s), 19,755 tokens, 57s elapsed
  batch 3/64: 25 records, 1 call(s), 20,957 tokens, 79s elapsed
  batch 4/64: 25 records, 1 call(s), 21,598 tokens, 101s elapsed
  batch 5/64: 25 records, 1 call(s), 22,192 tokens, 132s elapsed
  batch 6/64: 25 records, 1 call(s), 24,691 tokens, 164s elapsed
  batch 7/64: 25 records, 1 call(s), 20,351 tokens, 184s elapsed
  batch 8/64: 25 records, 1 call(s), 20,906 tokens, 205s elapsed
  batch 9/64: 25 records, 1 call(s), 22,763 tokens, 237s elapsed
  batch 10/64: 25 records, 1 call(s), 20,851 tokens, 258s elapsed
  batch 11/64: 25 records, 1 call(s), 20,490 tokens, 281s elapsed
  batch 12/64: 25 records, 1 call(s), 22,562 tokens, 315s elapsed
  batch 13/64: 25 records, 1 call(s), 22,076 tokens, 349s elapsed
  batch 14/64: 25 records, 1 call(s), 22,578 tokens, 382s el

In [35]:
classify.status()

full: 64 batches saved, 1600 records
